# Практика · Калібрування і вибір порога> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.md](homework.md) · Тест: [quiz.html](quiz.html)Дошка оголошень про вживані телефони. Дешевий фільтр відібрав підозрілі оголошення йсклав їх у чергу ручної перевірки — приблизно третина з них справді виявляється приманкою.Наше завдання — розібрати цю чергу автоматично.Що зробимо:1. Згенеруємо чергу з 6000 оголошень і навчимо логістичну регресію.2. Переконаємось, що `predict()` — це просто `predict_proba() >= 0.5`.3. Побудуємо криву «поріг → втрати в гривнях» і знайдемо оптимум.4. Порівняємо його з формулою `t* = C_FP / (C_FP + C_FN)`.5. Порахуємо діаграму надійності **руками** на кошиках і звіримо з `calibration_curve`.6. Зміряємо Brier score та ECE, теж спершу своїм кодом.7. Прожене чотири моделі через `CalibratedClassifierCV` обома методами й подивимось,   що станеться з AUC і що — з Brier.

## 1 · ДаніКожне оголошення описують шість ознак. Три з них — це фактично три погляди на однуобставину (ціна в гривнях, відносна ціна, відсоток знижки); це не помилка генерації,а типова ситуація реального датасету, і далі вона нам знадобиться.Мітку ми не вигадуємо руками: для кожного оголошення рахуємо справжню ймовірністьшахрайства й кидаємо монетку з цією ймовірністю. Тому «ідеальна» модель тут існує,і ми знаємо, наскільки вона добра.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_ads = 6000

# ознаки оголошення; розподіли підібрані так, щоб числа виглядали як справжні
price_ratio = rng.lognormal(0.0, 0.25, n_ads)              # ціна / медіанна ціна цієї моделі
account_age_days = rng.gamma(2.0, 90.0, n_ads) + 1         # вік акаунта продавця
photo_count = rng.poisson(4.0, n_ads)                      # скільки фото в оголошенні
# опис довший там, де більше фото: обидва показують старанність продавця
description_length = 40 + 30 * photo_count + rng.gamma(2.0, 25.0, n_ads)
price_uah = price_ratio * 8000 + rng.normal(0, 150, n_ads)  # та сама ціна, але в гривнях
discount_percent = (1 - price_ratio) * 100 + rng.normal(0, 2, n_ads)  # і вона ж як знижка

# справжня схильність до шахрайства: дешевше за ринок, новий акаунт, мало фото
true_logit = (2.1
              - 4.6 * (price_ratio - 1.0)
              - 1.0 * np.log(account_age_days / 30.0)
              - 0.18 * photo_count
              - 0.003 * description_length)
true_probability = 1 / (1 + np.exp(-true_logit))
# мітку кидаємо монеткою: так у даних є чесна випадковість, як у житті
is_fraud = (rng.random(n_ads) < true_probability).astype(int)

features = np.column_stack([price_ratio, account_age_days, photo_count,
                            description_length, price_uah, discount_percent])
feature_names = ["price_ratio", "account_age_days", "photo_count",
                 "description_length", "price_uah", "discount_percent"]

board = pd.DataFrame(features, columns=feature_names)
board["is_fraud"] = is_fraud
print("оголошень у черзі:", n_ads)
print("частка шахрайських:", round(is_fraud.mean(), 4))
board.head()

## 2 · Ділимо на навчальну й перевірочну частиниУсе, що ми далі міряємо, міряємо на перевірочній частині, якої модель не бачила.Поріг ми теж підбиратимемо на ній — і це навмисне спрощення для навчального зошита.У справжньому проєкті поріг обирають на **валідаційній** вибірці, а перевірочнувідкривають один раз наприкінці; інакше вийде те саме підглядання, про яке йшлосяв темі про підбір гіперпараметрів.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, is_fraud, test_size=0.35, random_state=42, stratify=is_fraud)

print("навчальних:", len(y_train), " перевірочних:", len(y_test))
print("частка шахрайських у перевірочній:", round(y_test.mean(), 4))

## 3 · `predict()` — це вже поріг 0.5Найкоротша клітинка зошита й найважливіша думка теми. Порівняємо готові вироки моделіз тим, що вийде, якщо взяти її ймовірності й самому порівняти їх із 0.5.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

proba_test = logreg.predict_proba(X_test)[:, 1]   # ймовірність класу «шахрай»
label_test = logreg.predict(X_test)               # готові вироки моделі

# те саме правило, тільки написане руками
manual_labels = (proba_test >= 0.5).astype(int)

assert np.array_equal(label_test, manual_labels), "predict() виявився чимось іншим!"
print("✅ predict() збігається з (predict_proba >= 0.5) на всіх", len(y_test), "оголошеннях")
print("розбіжностей:", int((label_test != manual_labels).sum()))

## 4 · Скільки коштує помилкаЗаблокувати чесне оголошення (FP) коштує майданчику ~200 грн: втрачена комісія йробота підтримки. Пропустити шахрайське (FN) коштує ~1800 грн: повернення грошейпокупцеві та розгляд скарги.Порахуємо сумарні втрати для кожного порогу від 0.01 до 0.99 і знайдемо найдешевший.

In [ ]:
COST_FALSE_ALARM = 200     # ціна хибної тривоги, грн
COST_MISS = 1800           # ціна пропуску, грн

def losses_at(scores, y_true, threshold):
    """Сумарні втрати в гривнях, якщо блокувати все, що не нижче порога."""
    predicted = (scores >= threshold).astype(int)
    false_alarms = int(((predicted == 1) & (y_true == 0)).sum())
    misses = int(((predicted == 0) & (y_true == 1)).sum())
    return COST_FALSE_ALARM * false_alarms + COST_MISS * misses

thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
loss_curve = np.array([losses_at(proba_test, y_test, t) for t in thresholds])

best_index = int(loss_curve.argmin())
best_threshold = thresholds[best_index]
formula_threshold = COST_FALSE_ALARM / (COST_FALSE_ALARM + COST_MISS)

print("поріг 0.50 (за замовчуванням):", losses_at(proba_test, y_test, 0.5), "грн")
print("найкращий поріг на даних:     ", best_threshold, "→", loss_curve[best_index], "грн")
print("поріг за формулою:            ", formula_threshold, "→",
      losses_at(proba_test, y_test, formula_threshold), "грн")
print("формула дорожча за фактичний мінімум на",
      round(100 * (losses_at(proba_test, y_test, formula_threshold) - loss_curve[best_index])
            / loss_curve[best_index], 2), "%")

Різниця між формулою й фактичним мінімумом — частки відсотка, і це головне: формулуможна порахувати **до** того, як побачив дані. Намалюймо криву цілком.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(thresholds, loss_curve, linewidth=2, label="сумарні втрати")
plt.axvline(best_threshold, linestyle="--", color="gray",
            label=f"мінімум на даних: {best_threshold}")
plt.axvline(formula_threshold, linestyle=":", color="crimson",
            label=f"формула: {formula_threshold}")
plt.axvline(0.5, linestyle="-.", color="orange", label="поріг за замовчуванням: 0.5")
plt.xlabel("поріг")
plt.ylabel("втрати, грн")
plt.title("Ціна порога на 2100 перевірочних оголошеннях")
plt.legend()
plt.tight_layout()
plt.show()

print("економія від правильного порога:",
      losses_at(proba_test, y_test, 0.5) - loss_curve[best_index], "грн")

## 5 · Три інші вимоги — три інші порогиЦіни помилок відомі не завжди. Подивимось, який поріг дають три поширені вимоги:задана повнота, квота на кількість хибних тривог і максимум F1.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def report_at(threshold):
    """Повнота, точність, F1 і кількість хибних тривог для одного порога."""
    predicted = (proba_test >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predicted, average="binary", zero_division=0)
    false_alarms = int(((predicted == 1) & (y_test == 0)).sum())
    return precision, recall, f1, false_alarms

# рахуємо звіт один раз для кожного порога, далі просто дивимось у нього
report = {t: report_at(t) for t in thresholds}

# 1) найвищий поріг, за якого повнота ще не нижча за 0.90
recall_rule = max(t for t in thresholds if report[t][1] >= 0.90)
# 2) найнижчий поріг, за якого хибних тривог не більше за 150
quota_rule = min(t for t in thresholds if report[t][3] <= 150)
# 3) поріг із найбільшою F1
f1_rule = thresholds[int(np.argmax([report[t][2] for t in thresholds]))]

rules = pd.DataFrame([
    ("нічого не обирали", 0.5),
    ("повнота >= 0.90", recall_rule),
    ("не більше 150 тривог", quota_rule),
    ("максимум F1", f1_rule),
    ("мінімум втрат", best_threshold),
], columns=["вимога", "поріг"])
rules["precision"] = [round(report_at(t)[0], 3) for t in rules["поріг"]]
rules["recall"] = [round(report_at(t)[1], 3) for t in rules["поріг"]]
rules["F1"] = [round(report_at(t)[2], 3) for t in rules["поріг"]]
rules["хибних тривог"] = [report_at(t)[3] for t in rules["поріг"]]
rules["втрати, грн"] = [losses_at(proba_test, y_test, t) for t in rules["поріг"]]
print(rules.to_string(index=False))

Одні й ті самі дані, одна й та сама модель — і пороги від краю до краю шкали. Жоденіз них не «правильніший»: вони відповідають на різні питання.## 6 · Діаграма надійності рукамиТепер перевіримо, чи можна читати числа моделі буквально. Ріжемо відрізок від 0 до 1на десять кошиків і в кожному порівнюємо середню обіцянку з фактичною часткою шахраїв.

In [ ]:
def reliability_by_hand(scores, y_true, n_bins=10):
    """Наш власний розрахунок діаграми надійності: середня оцінка й факт у кожному кошику."""
    bin_index = np.clip((scores * n_bins).astype(int), 0, n_bins - 1)
    mean_predicted, actual_share, sizes = [], [], []
    for b in range(n_bins):
        inside = bin_index == b
        if inside.sum() == 0:          # порожній кошик просто не дає точки
            continue
        mean_predicted.append(scores[inside].mean())
        actual_share.append(y_true[inside].mean())
        sizes.append(int(inside.sum()))
    return np.array(mean_predicted), np.array(actual_share), np.array(sizes)

predicted_by_bin, actual_by_bin, bin_sizes = reliability_by_hand(proba_test, y_test)

table = pd.DataFrame({
    "оголошень": bin_sizes,
    "середня оцінка": np.round(predicted_by_bin, 3),
    "фактична частка": np.round(actual_by_bin, 3),
    "розрив": np.round(np.abs(predicted_by_bin - actual_by_bin), 3),
})
print(table.to_string(index=False))

Те саме вміє `sklearn.calibration.calibration_curve`. Звіримо до останнього знака —усередині бібліотеки немає магії, там рівно той самий цикл по кошиках.

In [ ]:
from sklearn.calibration import calibration_curve

library_actual, library_predicted = calibration_curve(y_test, proba_test, n_bins=10)

assert np.allclose(library_predicted, predicted_by_bin), "середні оцінки розійшлися!"
assert np.allclose(library_actual, actual_by_bin), "фактичні частки розійшлися!"
print("✅ наш розрахунок збігається з calibration_curve у всіх",
      len(predicted_by_bin), "кошиках")

## 7 · Brier score і ECE**Brier** — середній квадрат різниці між оцінкою та фактом (0 або 1). Міряє одразудві речі: чи вміє модель розрізняти й чи чесні її числа.**ECE** — середній розрив «обіцяне мінус фактичне», зважений за розміром кошика.Міряє тільки калібрування.Обидва напишемо самі й звіримо Brier із бібліотечним.

In [ ]:
from sklearn.metrics import brier_score_loss, roc_auc_score

def brier_by_hand(scores, y_true):
    """Середній квадрат промаху: те саме, що середньоквадратична помилка для ймовірностей."""
    return float(np.mean((scores - y_true) ** 2))

def expected_calibration_error(scores, y_true, n_bins=10):
    """Середній розрив між обіцянкою й фактом, зважений за кількістю оголошень у кошику."""
    predicted, actual, sizes = reliability_by_hand(scores, y_true, n_bins)
    return float(np.sum(sizes / len(y_true) * np.abs(predicted - actual)))

assert np.isclose(brier_by_hand(proba_test, y_test),
                  brier_score_loss(y_test, proba_test)), "Brier розійшовся!"
print("✅ наш Brier збігається з brier_score_loss")
print("Brier логістичної регресії:", round(brier_by_hand(proba_test, y_test), 4))
print("ECE логістичної регресії:  ", round(expected_calibration_error(proba_test, y_test), 4))

# орієнтир: модель, яка всім називає частку шахраїв у черзі
constant_scores = np.full_like(proba_test, y_test.mean())
print("Brier моделі-константи:    ", round(brier_by_hand(constant_scores, y_test), 4),
      "— усе, що гірше, гірше за відмову класифікувати")

## 8 · Хто бреше і як самеНавчимо на тих самих даних чотири моделі й порівняємо їхні криві надійності.Дивимось на три числа окремо: AUC (чи вміє впорядковувати), Brier (загальна якістьімовірностей) і ECE (чи чесні числа).

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    "логістична регресія": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "випадковий ліс": RandomForestClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "наївний Баєс": GaussianNB(),
}

scores_by_model = {}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    scores_by_model[name] = scores
    rows.append((name,
                 round(roc_auc_score(y_test, scores), 4),
                 round(brier_by_hand(scores, y_test), 4),
                 round(expected_calibration_error(scores, y_test), 4),
                 round(scores.min(), 3), round(scores.max(), 3)))

comparison = pd.DataFrame(rows, columns=["модель", "AUC", "Brier", "ECE",
                                         "мін. оцінка", "макс. оцінка"])
print(comparison.to_string(index=False))

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], "--", color="gray", label="ідеальна калібровка")
for name, scores in scores_by_model.items():
    predicted, actual, _ = reliability_by_hand(scores, y_test)
    plt.plot(predicted, actual, marker="o", linewidth=1.8, label=name)
plt.xlabel("передбачена ймовірність")
plt.ylabel("фактична частка шахрайських")
plt.title("Діаграми надійності чотирьох моделей")
plt.legend()
plt.tight_layout()
plt.show()

print("AdaBoost тримає всі оцінки в діапазоні",
      round(scores_by_model["AdaBoost"].min(), 3), "–",
      round(scores_by_model["AdaBoost"].max(), 3))

Зверни увагу на AdaBoost: найкраща AUC із чотирьох — і найгірший ECE. Як сортувальниквін найкращий, як вимірювач імовірності — найгірший. Наївний Баєс псує числа інакше:він надто впевнений, бо вважає ціну в гривнях, відносну ціну й відсоток знижки трьоманезалежними доказами й рахує один і той самий доказ тричі.## 9 · Лікування: Платт та ізотонічна регресія`CalibratedClassifierCV` навчає поверх моделі другу, одновимірну: вона бере оцінку йперекладає її в чесну ймовірність. `method='sigmoid'` — це калібрування Платта,`method='isotonic'` — ізотонічна регресія. Параметр `cv=3` потрібен, щоб калібрувальникнавчався не на тих даних, на яких навчена базова модель: у кожній згортці базова модельбачить дві третини, а калібрувальник — відкладену третину.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

def make_model(name):
    """Свіжий, ненавчений екземпляр — CalibratedClassifierCV навчить його сам."""
    if name == "логістична регресія":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    if name == "AdaBoost":
        return AdaBoostClassifier(n_estimators=100, random_state=42)
    return GaussianNB()

results = []
for name in ["логістична регресія", "AdaBoost", "наївний Баєс"]:
    raw_scores = scores_by_model[name]
    row = {"модель": name,
           "AUC як є": round(roc_auc_score(y_test, raw_scores), 4),
           "Brier як є": round(brier_by_hand(raw_scores, y_test), 4),
           "ECE як є": round(expected_calibration_error(raw_scores, y_test), 4)}
    for method, label in [("sigmoid", "Платт"), ("isotonic", "ізотон.")]:
        calibrated = CalibratedClassifierCV(make_model(name), method=method, cv=3)
        calibrated.fit(X_train, y_train)
        fixed_scores = calibrated.predict_proba(X_test)[:, 1]
        row[f"AUC {label}"] = round(roc_auc_score(y_test, fixed_scores), 4)
        row[f"Brier {label}"] = round(brier_by_hand(fixed_scores, y_test), 4)
        row[f"ECE {label}"] = round(expected_calibration_error(fixed_scores, y_test), 4)
        if name == "AdaBoost":
            scores_by_model[f"AdaBoost · {label}"] = fixed_scores
    results.append(row)

calibration_table = pd.DataFrame(results)
print(calibration_table[["модель", "AUC як є", "AUC Платт", "AUC ізотон."]].to_string(index=False))
print()
print(calibration_table[["модель", "Brier як є", "Brier Платт", "Brier ізотон."]].to_string(index=False))
print()
print(calibration_table[["модель", "ECE як є", "ECE Платт", "ECE ізотон."]].to_string(index=False))

Три речі, які варто прочитати в цих таблицях.1. **AUC майже не змінилась.** Калібрування — монотонне перетворення, воно не може   переставити оголошення місцями. Здатність розрізняти воно не покращує й не псує.2. **Brier і ECE впали там, де було зламано.** У AdaBoost — різко.3. **У логістичної регресії не змінилось нічого.** Калібрувати калібровану модель —   марна робота. Спершу міряй, потім лікуй.Подивимось на AdaBoost до й після.

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], "--", color="gray", label="ідеальна калібровка")
for name in ["AdaBoost", "AdaBoost · Платт", "AdaBoost · ізотон."]:
    predicted, actual, _ = reliability_by_hand(scores_by_model[name], y_test)
    plt.plot(predicted, actual, marker="o", linewidth=1.8, label=name)
plt.xlabel("передбачена ймовірність")
plt.ylabel("фактична частка шахрайських")
plt.title("AdaBoost до й після калібрування")
plt.legend()
plt.tight_layout()
plt.show()

before = scores_by_model["AdaBoost"]
after = scores_by_model["AdaBoost · Платт"]
print("діапазон оцінок до калібрування: ", round(before.min(), 3), "–", round(before.max(), 3))
print("після калібрування Платта:       ", round(after.min(), 3), "–", round(after.max(), 3))

## 10 · Навіщо все це було: поріг на некаліброваній моделіОстанній експеримент теми. Візьмемо формулу `t* = C_FP / (C_FP + C_FN) = 0.1` ізастосуємо її до сирого AdaBoost, який ніколи не видає оцінок нижчих за 0.23.

In [ ]:
raw_boost = scores_by_model["AdaBoost"]
fixed_boost = scores_by_model["AdaBoost · Платт"]

print("сирий AdaBoost на порозі 0.1: заблоковано",
      int((raw_boost >= formula_threshold).sum()), "оголошень із", len(y_test),
      "→", losses_at(raw_boost, y_test, formula_threshold), "грн")
print("він же після калібрування:    заблоковано",
      int((fixed_boost >= formula_threshold).sum()), "оголошень із", len(y_test),
      "→", losses_at(fixed_boost, y_test, formula_threshold), "грн")

# а якщо поріг не рахувати формулою, а підбирати по даних — обидві версії однакові
raw_best = min(losses_at(raw_boost, y_test, t) for t in thresholds)
fixed_best = min(losses_at(fixed_boost, y_test, t) for t in thresholds)
print()
print("найкращі можливі втрати, сирий:      ", raw_best, "грн")
print("найкращі можливі втрати, калібрований:", fixed_best, "грн")
print("порядок той самий, тому й мінімум майже той самий —",
      "калібрування потрібне саме тоді, коли поріг РАХУЮТЬ, а не підбирають")

---## Завдання### 🟢 Рівень 1 — БазаЗміни ціни помилок на `COST_FALSE_ALARM = 500` і `COST_MISS = 1000` і перерахуйвсе з розділу 4. Куди поїхав оптимальний поріг і чому саме туди? Звір фактичниймінімум із формулою.**Зроблено, якщо:** названо новий поріг за формулою, новий фактичний мінімум іпояснено словами, чому поріг зріс.### 🟡 Рівень 2 — ПлюсПорахуй ECE для логістичної регресії на 5, 10, 20 і 50 кошиках. Побудуй графік«кількість кошиків → ECE» і поясни, чому число росте. Що це означає для порівнянняECE двох моделей із різних статей?**Зроблено, якщо:** є графік із чотирьох точок і висновок про те, що ECE беззазначеної кількості кошиків не має сенсу.### 🔴 Рівень 3 — ВикликРеалізуй ізотонічну регресію самостійно — алгоритмом PAVA (pool adjacent violators):відсортуй оцінки, поклади кожну мітку окремим блоком і, доки знайдеться пара сусідніхблоків, у якої лівий не менший за правий, зливай їх у один із середнім значенням.Навчи свою реалізацію на половині перевірочної вибірки, застосуй до другої половиний порівняй Brier із `IsotonicRegression` зі `sklearn`.**Зроблено, якщо:** `np.allclose` між твоїми та бібліотечними ймовірностями проходитьіз точністю 0.01, а різниця в Brier менша за 0.001.### Підказки- У рівні 1 формула дасть поріг більший за 0.5 — це не помилка, а наслідок того, що  хибна тривога стала дорожчою за половину пропуску.- У рівні 2 подумай, скільки оголошень лишається в одному кошику при 50 кошиках і  наскільки стабільна фактична частка, порахована на такій жменьці.- У рівні 3 зручно тримати три паралельні списки: значення блоку, його вагу  (скільки оголошень злилось) і праву межу за оцінкою.